In [2]:
"""
fidelity_comparison.py - Compare QPA circuit fidelity with and without noise

This script compares the fidelity of QPA (Quantum Phase Algorithm) circuits with
and without noise, using either trotterized or exact statevector as the target state.
It generates plots showing how fidelity changes with different noise strengths (epsilon).

Key Features:
- Implements Ising model simulation with trotterized evolution
- Supports both exact and trotterized statevector preparation
- Uses Aer simulator with custom noise models
- Compares fidelity of Ising circuit vs QPA circuit
- Supports configurable noise strength (epsilon) sweeps
- Generates plots comparing fidelity vs noise strength

How to Run:
python fidelity_comparison.py [OPTIONS]

Required Dependencies:
- qiskit
- numpy
- pandas
- matplotlib
- scipy

Example Usage:
1. Basic run with default parameters:
   python fidelity_comparison.py

2. Run with specific parameters:
   python estimation_scripts/fidelity_comparison.py --shots 1024 --t 3.0 --J 1.0 --h 1.0 --steps 5       --eps_min 0.0 --eps_max 0.1 --eps_steps 20 --output output.png --trotter True

3. Run with different noise range:
   python fidelity_comparison.py --eps_min 0.0 --eps_max 0.2 --eps_steps 20

The script will generate:
- A plot comparing Ising vs QPA circuit fidelities
- CSV files containing the raw fidelity data
- Circuit diagrams (if requested)

Note: The script uses Aer simulator with custom noise models that include:
- Depolarizing errors for 1-qubit gates (id_noisy, rx)
- Depolarizing errors for 2-qubit gates (rzz)
- Depolarizing errors for 3-qubit gates (cswap)
- Readout errors
"""

import numpy as np
import pandas as pd
import argparse
from tqdm import tqdm
import os
import logging
from datetime import datetime

from scipy.linalg import expm
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
from qiskit.quantum_info import (
    Statevector,
    SparsePauliOp
)
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.noise import (
    NoiseModel,
    ReadoutError,
    depolarizing_error,
)
class ising_class:
    """
    Class for generating Ising model circuits and statevectors.
    
    Args:
        d: Number of qubits per register
        steps: Number of Trotter steps
        t: Evolution time
        J: Interaction strength
        h: Transverse field strength
    """
    def __init__(self, d, steps, t, J, h):
        self.d = d
        self.steps = steps
        self.t = t
        self.J = J
        self.h = h

    def get_trotterized_ising_circuit(self):
        """
        Returns a QuantumCircuit implementing a trotterized Ising evolution.
        """
        dt = self.t / self.steps
        qc = QuantumCircuit(self.d)

        for _ in range(self.steps):
            # Apply ZZ interactions (Z_i Z_{i+1})
            for i in range(self.d - 1):
                qc.cx(i, i + 1)
                qc.rz(-2 * self.J * dt, i + 1)
                qc.cx(i, i + 1)

            # Apply transverse field X terms (X_i)
            for i in range(self.d):
                qc.rx(-2 * self.h * dt, i)

        return qc

    def apply_ising_to_registers(self, qc,start):
        """
        Apply trotterized Ising circuit to registers q1, q2, q3 in a 4d-sized register circuit.
        """
        ising = self.get_trotterized_ising_circuit()
        for reg in [1, 2, 3]:
            for gate, qargs, cargs in ising.data:
                mapped_qargs = [qc.qubits[start+(reg-1) * self.d + ising.qubits.index(q)] for q in qargs]
                qc.append(gate, mapped_qargs, cargs)
        return qc
    def get_trotterized_ising_statevector(self):
        """
        Returns the statevector from the trotterized Ising evolution.
        """
        qc = self.get_trotterized_ising_circuit()
        qc.save_statevector()
        simulator = AerSimulator()
        result = simulator.run(transpile(qc, simulator)).result()
        return result.get_statevector()

def get_QPA_circuit(d, ising_circuit, nqpa):
    """
    Returns a simplified QPA circuit with single control qubit and single QPA sequence.
    
    Args:
        d: Number of qubits per register
        ising_circuit: ising_class instance
        
    Returns:
        QuantumCircuit implementing the QPA sequence
    """
    # Single control qubit
    cr = ClassicalRegister(1, name='control')
    qr_all = QuantumRegister(3*d + 1)
    
    # Initialize quantum circuit
    qc = QuantumCircuit(qr_all, cr)
    
    # Apply Ising circuit to registers
    qc = ising_circuit.apply_ising_to_registers(qc, start=1)
    if nqpa==0:
        return qc

    assert nqpa==1 #Cannot have a different value for this circuit for now
    
    # Single QPA sequence
    qc.h(0)      # Apply Hadamard
    for k in range(d):
        qc.cswap(0, k+1, k+d+1)  # Control qubit 0, target q1 and q2
    qc.h(0)      # Apply second Hadamard
    
    return qc

def compute_exact_statevector(k, t, J, h):
    """
    Compute the exact statevector for the Ising Hamiltonian.
    
    Args:
        k: Number of qubits
        t: Evolution time
        J: Interaction strength
        h: Transverse field strength
        
    Returns:
        Statevector of the evolved state
    """
    dim = 2**k
    I = np.eye(2)
    X = np.array([[0, 1], [1, 0]])
    Z = np.array([[1, 0], [0, -1]])

    def kron_n(*ops):
        """Kronecker product of operators."""
        out = ops[0]
        for op in ops[1:]:
            out = np.kron(out, op)
        return out

    # Build Hamiltonian
    H = np.zeros((dim, dim), dtype=complex)
    for q in range(k - 1):
        ops = [I] * k
        ops[q] = Z
        ops[q + 1] = Z
        H += -J * kron_n(*ops)
    
    for q in range(k):
        ops = [I] * k
        ops[q] = X
        H += -h * kron_n(*ops)

    # Initial state |000>
    psi0 = np.zeros(dim, dtype=complex)
    psi0[0] = 1.0

    # Time evolution: U = exp(-iHt)
    U = expm(-1j * H * t)
    psi_final = U @ psi0

    return Statevector(psi_final)

def get_projector_for_each_z(state, z, k):
    """
    Create a projector for the specified control qubit state.
    
    Args:
        state: Statevector to project
        z: Control qubit state (0 or 1)
        
    Returns:
        SparsePauliOp representing the projector
    """
    fidelity_operator = SparsePauliOp.from_operator(state.to_operator())
    identity_op = SparsePauliOp(["I" * k])
    if z == 0:
        # For z=0: Swap q3 to q2 (q2 is the target of the projection)
        control_state = Statevector([1,0])
        control_projector = SparsePauliOp.from_operator(control_state.to_operator())
        return identity_op.tensor(fidelity_operator).tensor(identity_op).tensor(control_projector)
    else:
        assert z == 1
        # For z=1: Keep q3 (q3 is the target of the projection)
        control_state = Statevector([0,1])
        control_projector = SparsePauliOp.from_operator(control_state.to_operator())
        return fidelity_operator.tensor(identity_op).tensor(identity_op).tensor(control_projector)

def estimate_fidelity(qc, state, estimator, k):
    """
    Estimate the fidelity using two projectors (z=0 and z=1).
    
    Args:
        qc: Quantum circuit to estimate
        state: Statevector to project
        backend: Backend to use
        shots: Number of shots
        
    Returns:
        Tuple of (fidelity_0, fidelity_1, total_fidelity)
    """
    # Create projectors
    projector_0 = get_projector_for_each_z(state, z=0, k=k)
    projector_1 = get_projector_for_each_z(state, z=1, k=k)
    
    # Apply layout
    layout = qc.layout
    observable_0 = projector_0.apply_layout(layout)
    observable_1 = projector_1.apply_layout(layout)
    
    # Run estimation
    job_0 = estimator.run([(qc, observable_0, None)]).result()
    job_1 = estimator.run([(qc, observable_1, None)]).result()
    
    # Calculate fidelities
    fidelity_0 = job_0[0].data.evs
    fidelity_1 = job_1[0].data.evs
    total_fidelity = fidelity_0 + fidelity_1
    
    return fidelity_0, fidelity_1, total_fidelity


# Function to create noise model
def create_noise_model(eps):
    """Create a noise model with depolarizing errors."""
    noise_model = NoiseModel()
    # Add depolarizing error for single qubit gates
    noise_model.add_all_qubit_quantum_error(depolarizing_error(eps, 1), ['id_noisy', 'rx'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(eps, 2), ['rzz'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(eps, 3), ['cswap'])
    readout_error = ReadoutError([[1 - eps, eps], [eps, 1 - eps]])
    noise_model.add_all_qubit_readout_error(readout_error)
    return noise_model



In [ ]:

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

k=2
shots = 1024 
t = 3.0 
J = 1.0 
h = 1.0 
steps = 5 
eps_min = 0.0 
eps_max = 0.1 
eps_steps = 20 
output = "output.png" 
trotterized_fidelity = True


# Create output directory
os.makedirs("data", exist_ok=True)

# Log parameters
logging.info(f"Starting simulation with parameters:")
logging.info(f"Shots: {shots}")
logging.info(f"Evolution time: {t}")
logging.info(f"Interaction strength: {J}")
logging.info(f"Transverse field: {h}")
logging.info(f"Trotter steps: {steps}")
logging.info(f"Epsilon range: {eps_min} to {eps_max}")
logging.info(f"Epsilon steps: {eps_steps}")
logging.info(f"Using trotterized state: {trotterized_fidelity}")



2025-06-12 14:49:24,390 - INFO - Starting simulation with parameters:
2025-06-12 14:49:24,391 - INFO - Shots: 1024
2025-06-12 14:49:24,391 - INFO - Evolution time: 3.0
2025-06-12 14:49:24,392 - INFO - Interaction strength: 1.0
2025-06-12 14:49:24,392 - INFO - Transverse field: 1.0
2025-06-12 14:49:24,392 - INFO - Trotter steps: 5
2025-06-12 14:49:24,392 - INFO - Epsilon range: 0.0 to 0.1
2025-06-12 14:49:24,393 - INFO - Epsilon steps: 20
2025-06-12 14:49:24,393 - INFO - Using trotterized state: True
/tmp/ipykernel_1173991/522267657.py:36: DeprecationWarning: The "ibm_quantum" channel option is deprecated and will be sunset on 1 July. After this date, "ibm_cloud", "ibm_quantum_platform", and "local" will be the only valid channels. Open Plan users should migrate now.  All other users should review the migration guide (https://quantum.cloud.ibm.com/docs/migration-guides/classic-iqp-to-cloud-iqp)to learn when to migrate.
  service = QiskitRuntimeService()
qiskit_runtime_service.__init__:I

NameError: name 'FakeSherbrooke' is not defined

In [ ]:
# Create ising instance
ising = ising_class(k, steps, t, J, h)  # Using fixed k=2

# Get target state
if trotterized_fidelity:
    target_state = ising.get_trotterized_ising_statevector()
else:
    target_state = compute_exact_statevector(k, t, J, h)  # Using fixed k=2

# Arrays to store results
epsilons = np.linspace(eps_min, eps_max, eps_steps)
ising_fidelities = []
qpa_fidelities = []

# Create circuits
qc_ising = get_QPA_circuit(k, ising, 0)
qc_qpa = get_QPA_circuit(k, ising, 1)

# Process each epsilon value with progress bar
for epsilon in tqdm(epsilons, desc="Processing epsilon values", unit="eps"):
    logging.info(f"\nProcessing epsilon = {epsilon:.3f}")
    
    # Create noise model
    noise_model = create_noise_model(epsilon)
    estimator = AerEstimator(options={
        'backend_options': {
            'noise_model': noise_model,
            'shots': shots
        }
    })
    # Estimate Ising circuit fidelity
    _,_,fidelity_ising = estimate_fidelity(qc_ising, target_state, estimator, k)
    ising_fidelities.append(fidelity_ising)
    logging.info(f"Ising fidelity: {fidelity_ising:.4f}")
    
    # Estimate QPA circuit fidelity
    _,_,fidelity_qpa = estimate_fidelity(qc_qpa, target_state, estimator, k)
    qpa_fidelities.append(fidelity_qpa)
    logging.info(f"QPA fidelity: {fidelity_qpa:.4f}")



2025-06-11 11:48:29,920 - INFO - Pass: ContainsInstruction - 0.00906 (ms)
2025-06-11 11:48:29,921 - INFO - Pass: UnitarySynthesis - 0.00715 (ms)
2025-06-11 11:48:29,921 - INFO - Pass: HighLevelSynthesis - 0.00596 (ms)
2025-06-11 11:48:29,922 - INFO - Pass: BasisTranslator - 0.06962 (ms)
2025-06-11 11:48:29,922 - INFO - Pass: ElidePermutations - 0.00811 (ms)
2025-06-11 11:48:29,922 - INFO - Pass: RemoveDiagonalGatesBeforeMeasure - 0.01335 (ms)
2025-06-11 11:48:29,922 - INFO - Pass: RemoveIdentityEquivalent - 0.00811 (ms)
2025-06-11 11:48:29,923 - INFO - Pass: InverseCancellation - 0.05674 (ms)
2025-06-11 11:48:29,923 - INFO - Pass: ContractIdleWiresInControlFlow - 0.00310 (ms)
2025-06-11 11:48:29,923 - INFO - Pass: CommutativeCancellation - 0.03719 (ms)
2025-06-11 11:48:29,924 - INFO - Pass: ConsolidateBlocks - 0.09990 (ms)
2025-06-11 11:48:29,924 - INFO - Pass: Split2QUnitaries - 0.03052 (ms)
2025-06-11 11:48:29,924 - INFO - Pass: UnitarySynthesis - 0.00453 (ms)
2025-06-11 11:48:29,924

In [ ]:
# Save raw data
df = pd.DataFrame({
    'epsilon': epsilons,
    'ising_fidelity': ising_fidelities,
    'qpa_fidelity': qpa_fidelities
})
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
data_file = f"data/fidelity_comparison_data_{timestamp}.csv"
df.to_csv(data_file, index=False)
logging.info(f"Saved raw data to {data_file}")

# Generate plot
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(epsilons, ising_fidelities, 'o-', label='Ising')
plt.plot(epsilons, qpa_fidelities, 'x-', label='QPA')
plt.xlabel('Noise Strength (ε)')
plt.ylabel('Fidelity')
plt.title('Fidelity Comparison: Ising vs QPA')
plt.legend()
plt.grid(True)
plt.savefig(args.output)
logging.info(f"Saved plot to {args.output}")
plt.close()